In [ ]:
!git clone https://github.com/Truong5724/machine-unlearning

Cloning into 'machine-unlearning'...
remote: Enumerating objects: 791, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 791 (delta 110), reused 109 (delta 93), pack-reused 639 (from 2)
Receiving objects: 100% (791/791), 44.86 MiB | 18.49 MiB/s, done.
Resolving deltas: 100% (443/443), done.


In [ ]:
%cd ..

/


In [ ]:
%cd content/machine-unlearning


/content/machine-unlearning


In [ ]:
%cd datasets/UTKFace

/content/machine-unlearning/datasets/UTKFace


In [ ]:
!pip install kaggle

In [ ]:
!mkdir -p ~/.kaggle

In [ ]:
%%writefile ~/.kaggle/kaggle.json
{
  "username":"nguyendinhtri",
  "key":"KGAT_2b37de79502159598cde433ad1de8c02"
}

Writing /root/.kaggle/kaggle.json


In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
100% 331M/331M [00:17<00:00, 20.2MB/s]



In [ ]:
!unzip utkface-new.zip

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  inflating: utkface_aligned_cropped/crop_part1/34_1_0_20170109004755204.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_0_20170111182452832.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_1_20170103230340961.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_1_20170104011329697.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_1_20170104165020320.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_1_20170108230211421.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_2_20170104022134829.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_2_20170104023010725.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_2_20170104172537171.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_2_20170104201443273.jpg.chip.jpg  
  inflating: utkface_aligned_cropped/crop_part1/34_1_2_20170104

In [ ]:
!pip install h5py tqdm Pillow

In [ ]:
!python prepare_data_ver2.py --img_dir UTKFace


CHUẨN BỊ UTKFACE DATASET
Task: gender
Target size: (64, 64)
Train ratio: 0.8
Age bins: [0, 18, 60, 117] → 3 classes

🔍 Đang tìm ảnh...
✅ Tìm thấy 23708 ảnh

🔍 Đang parse filenames...
Parsing: 100% 23708/23708 [00:00<00:00, 179711.60it/s]
✅ Parse thành công 23705/23708 ảnh

PHÂN BỐ GENDER
  Female               :  12391 samples ( 52.3%)
  Male                 :  11314 samples ( 47.7%)

PHÂN BỐ AGE BIN (3-class)
  Young (0-17)         :   4233 samples ( 17.9%)
  Adult (18-59)        :  16784 samples ( 70.8%)
  Senior (60+)         :   2688 samples ( 11.3%)

PHÂN BỐ RACE (5-class)
  White                :  10078 samples ( 42.5%)
  Black                :   4526 samples ( 19.1%)
  Asian                :   3434 samples ( 14.5%)
  Indian               :   3975 samples ( 16.8%)
  Others               :   1692 samples (  7.1%)

🔀 Shuffling data...

📊 Split:
   Train: 18964 samples
   Test:  4741 samples

📦 Tạo utkface_train.h5...
Đang lưu 18964 ảnh vào HDF5...
Processing: 100% 19/19 [00:43<00:0

In [ ]:
%cd ..

/content/machine-unlearning


In [ ]:
%cd ..

In [ ]:
!touch datasets/__init__.py
!touch datasets/UTKFace/__init__.py

In [ ]:
#!rm -rf containers
#print('Đã xóa thư mục containers.')

Đã xóa thư mục containers.


In [ ]:
!bash example-scripts/utkface-sharding/init_multitask.sh 1

🚀 Init SISA UTKFace - 1 shards + 3 scenarios
📦 Creating shards and request files...
✅ Created 1 uniform shards
✅ Created requestfile:0 with 0 requests
✅ Created requestfile for 0 samples
✅ Created requestfile:100 with 100 requests
✅ Created requestfile for 100 samples
✅ Created requestfile:500 with 500 requests
✅ Created requestfile for 500 samples
✅ Init completed successfully!
Next step: ./train.sh 1


In [ ]:
import numpy as np

shards = np.load("containers/utkface/splitfile.npy", allow_pickle=True)
requests_100 = np.load("containers/utkface/requestfile:100.npy", allow_pickle=True)
requests_500 = np.load("containers/utkface/requestfile:500.npy", allow_pickle=True)

retained_100 = np.sort(np.setdiff1d(shards[0], requests_100[0], assume_unique=True))
retained_500 = np.sort(np.setdiff1d(shards[0], requests_500[0], assume_unique=True))

# Đếm số "đoạn liên tục" (contiguous run) trong mỗi tập
def count_runs(arr):
    return 1 + np.sum(np.diff(arr) != 1)

print("Retained 100 - số đoạn liên tục:", count_runs(retained_100))
print("Retained 500 - số đoạn liên tục:", count_runs(retained_500))

Retained 100 - số đoạn liên tục: 101
Retained 500 - số đoạn liên tục: 475


#TRƯỜNG HỢP 1 SHARD 3 SLICE

In [ ]:
!rm -rf containers
print('Đã xóa thư mục containers.')

In [ ]:
!bash example-scripts/utkface-sharding/init_multitask.sh 1

In [ ]:
!bash example-scripts/utkface-sharding/train_multitask.sh 1

In [ ]:
!bash example-scripts/utkface-sharding/predict_multitask.sh 1 0

In [ ]:
!bash example-scripts/utkface-sharding/predict_multitask.sh 1 100

In [ ]:
!bash example-scripts/utkface-sharding/predict_multitask.sh 1 500

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 0

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 100

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 500

#TRƯỜNG HỢP 1 SHARD 1 SLICE

In [ ]:
!bash example-scripts/utkface-sharding/train_multitask.sh 1

🚀 TRAINING SISA UTKFace MULTITASK
Shards     : 1
Scenarios  : 0
100
500
Epochs     : 30
Batch size : 64
Slices     : 1

🔄 SCENARIO label=0
→ Training Shard 0 / 1 (label=0)
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
Original: 18964
Removed : 0
Retained: 18964
Label    : 0
Shard    : 0
Original : 18964
Removed  : 0
Retained : 18964
[info] Using weighted CrossEntropyLoss for race
Shard 0: 100%|██████████| 1/1 [04:22<00:00, 262.52s/it]
[Shard 0][Slice 0][Epoch 1] loss=797.2619 gender=67.4% age=74.7% race=47.7% mean=63.2%
    Time: total=9.73s | batch=2.76s | overhead=6.97s
[Shard 0][Slice 0][Epoch 2] loss=670.9194 gender=77.8% age=79.5% race=55.3% mean=70.9%
    Time: total=8.42s | batch=2.29s | overhead=6.13s
[Shard 0][Slice 0][Epoch 3] loss=593.4512 gender=82.0% age=82.2% race=60.8% mean=75.0%
    Time: total=8.75s | batch=2.33s | overhead=6.42s
[Shard 0][Slice 0][Epoch 4] loss=547.6460 gender=83.7% age=83.5% race=64.6% mean=77.3%
    Time: total=8.95s | batch=2.38s | overhead=6.57s


In [ ]:
!bash example-scripts/utkface-sharding/predict_multitask.sh 1 0

PREDICTION - UTKFACE MULTITASK
Shards     : 1
Label      : 0
Batch size : 128

→ Shard 1/1 (label=0)
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
Shard 0 test metrics (all 3 tasks)
gender: acc=89.41% prec=89.52% bacc=89.56% f1=89.41%
age   : acc=87.81% prec=84.11% bacc=78.51% f1=80.69%
race  : acc=72.87% prec=66.77% bacc=67.91% f1=65.47%
✅ Shard 0 prediction done
example-scripts/utkface-sharding/predict_multitask.sh: line 59: total: command not found


In [ ]:
!bash example-scripts/utkface-sharding/predict_multitask.sh 1 100

PREDICTION - UTKFACE MULTITASK
Shards     : 1
Label      : 100
Batch size : 128

→ Shard 1/1 (label=100)
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
Shard 0 test metrics (all 3 tasks)
gender: acc=88.84% prec=89.57% bacc=88.56% f1=88.72%
age   : acc=85.85% prec=91.79% bacc=63.61% f1=69.44%
race  : acc=73.76% prec=67.90% bacc=71.27% f1=68.32%
✅ Shard 0 prediction done
example-scripts/utkface-sharding/predict_multitask.sh: line 59: total: command not found


In [ ]:
!bash example-scripts/utkface-sharding/predict_multitask.sh 1 500

PREDICTION - UTKFACE MULTITASK
Shards     : 1
Label      : 500
Batch size : 128

→ Shard 1/1 (label=500)
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
Shard 0 test metrics (all 3 tasks)
gender: acc=82.47% prec=86.21% bacc=81.77% f1=81.76%
age   : acc=88.84% prec=85.60% bacc=78.81% f1=81.78%
race  : acc=72.41% prec=67.70% bacc=65.19% f1=64.82%
✅ Shard 0 prediction done
example-scripts/utkface-sharding/predict_multitask.sh: line 59: total: command not found


#TRƯỜNG HỢP 1 SHARD 1 SLICE

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 0

📊 DATA SUMMARY - UTKFace SISA
Shards : 1
Label  : 0
🔍 Checking outputs...
✅ All outputs exist.
📈 Computing test set metrics...
Test metrics:
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
UTKFACE MULTITASK EVALUATION (TEST SET - SISA)
Container : utkface
Label     : 0
Shards    : 1
Strategy  : uniform (majority vote)
----------------------------------------------------------------------
{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
gender: acc= 89.41% | macro-prec= 89.52% | macro-recall= 89.56% | macro-f1= 89.41%
  Class metrics:
dict_keys(['Male', 'Female'])
    Male      : prec= 92.98% | recall= 86.21% | f1= 89.47% | support=2473
    Female    : prec= 86.07% | recall= 92.90% | f1= 89.36% | support=2268

{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
age   : acc= 87

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 100

📊 DATA SUMMARY - UTKFace SISA
Shards : 1
Label  : 100
🔍 Checking outputs...
✅ All outputs exist.
📈 Computing test set metrics...
Test metrics:
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
UTKFACE MULTITASK EVALUATION (TEST SET - SISA)
Container : utkface
Label     : 100
Shards    : 1
Strategy  : uniform (majority vote)
----------------------------------------------------------------------
{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
gender: acc= 88.84% | macro-prec= 89.57% | macro-recall= 88.56% | macro-f1= 88.72%
  Class metrics:
dict_keys(['Male', 'Female'])
    Male      : prec= 85.17% | recall= 95.19% | f1= 89.90% | support=2473
    Female    : prec= 93.98% | recall= 81.92% | f1= 87.54% | support=2268

{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
age   : acc

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 500

📊 DATA SUMMARY - UTKFace SISA
Shards : 1
Label  : 500
🔍 Checking outputs...
✅ All outputs exist.
📈 Computing test set metrics...
Test metrics:
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
UTKFACE MULTITASK EVALUATION (TEST SET - SISA)
Container : utkface
Label     : 500
Shards    : 1
Strategy  : uniform (majority vote)
----------------------------------------------------------------------
{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
gender: acc= 82.47% | macro-prec= 86.21% | macro-recall= 81.77% | macro-f1= 81.76%
  Class metrics:
dict_keys(['Male', 'Female'])
    Male      : prec= 75.61% | recall= 98.02% | f1= 85.37% | support=2473
    Female    : prec= 96.81% | recall= 65.52% | f1= 78.15% | support=2268

{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
age   : acc

#TRƯỜNG HỢP 1 SHARD 5 SLICES

In [ ]:
!rm -rf containers
print('Đã xóa thư mục containers.')

Đã xóa thư mục containers.


In [ ]:
!bash example-scripts/utkface-sharding/init_multitask.sh 1

🚀 Init SISA UTKFace - 1 shards + 3 scenarios
📦 Creating shards and request files...
✅ Created 1 uniform shards
✅ Created requestfile:0 with 0 requests
✅ Created requestfile for 0 samples
✅ Created requestfile:100 with 100 requests
✅ Created requestfile for 100 samples
✅ Created requestfile:500 with 500 requests
✅ Created requestfile for 500 samples
✅ Init completed successfully!
Next step: ./train.sh 1


In [ ]:
!bash example-scripts/utkface-sharding/train_multitask.sh 1

🚀 TRAINING SISA UTKFace MULTITASK
Shards     : 1
Scenarios  : 0
100
500
Epochs     : 30
Batch size : 64
Slices     : 5

🔄 SCENARIO label=0
→ Training Shard 0 / 1 (label=0)
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
Original: 18964
Removed : 0
Retained: 18964
Label    : 0
Shard    : 0
Original : 18964
Removed  : 0
Retained : 18964
[info] Using weighted CrossEntropyLoss for race
Shard 0:  20%|██        | 1/5 [00:17<01:09, 17.41s/it][Shard 0][Slice 0][Epoch 1] loss=176.3691 gender=59.4% age=70.2% race=43.9% mean=57.8%
    Time: total=1.99s | batch=0.73s | overhead=1.26s
[Shard 0][Slice 0][Epoch 2] loss=163.9953 gender=66.1% age=73.2% race=46.1% mean=61.8%
    Time: total=1.65s | batch=0.45s | overhead=1.20s
[Shard 0][Slice 0][Epoch 3] loss=156.4227 gender=68.4% age=75.1% race=49.4% mean=64.3%
    Time: total=1.91s | batch=0.50s | overhead=1.41s
[Shard 0][Slice 0][Epoch 4] loss=150.3749 gender=71.5% age=76.7% race=50.3% mean=66.2%
    Time: total=1.96s | batch=0.51s | overhead=1.45s
[S

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 0

📊 DATA SUMMARY - UTKFace SISA
Shards : 1
Label  : 0
🔍 Checking outputs...
✅ All outputs exist.
📈 Computing test set metrics...
Test metrics:
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
UTKFACE MULTITASK EVALUATION (TEST SET - SISA)
Container : utkface
Label     : 0
Shards    : 1
Strategy  : uniform (majority vote)
----------------------------------------------------------------------
{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
gender: acc= 88.63% | macro-prec= 88.86% | macro-recall= 88.77% | macro-f1= 88.63%
  Class metrics:
dict_keys(['Male', 'Female'])
    Male      : prec= 92.79% | recall= 84.57% | f1= 88.49% | support=2449
    Female    : prec= 84.93% | recall= 92.98% | f1= 88.77% | support=2292

{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
age   : acc= 87

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 100

📊 DATA SUMMARY - UTKFace SISA
Shards : 1
Label  : 100
🔍 Checking outputs...
✅ All outputs exist.
📈 Computing test set metrics...
Test metrics:
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
UTKFACE MULTITASK EVALUATION (TEST SET - SISA)
Container : utkface
Label     : 100
Shards    : 1
Strategy  : uniform (majority vote)
----------------------------------------------------------------------
{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
gender: acc= 88.50% | macro-prec= 88.84% | macro-recall= 88.67% | macro-f1= 88.50%
  Class metrics:
dict_keys(['Male', 'Female'])
    Male      : prec= 93.35% | recall= 83.71% | f1= 88.27% | support=2449
    Female    : prec= 84.32% | recall= 93.63% | f1= 88.73% | support=2292

{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
age   : acc

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 500

📊 DATA SUMMARY - UTKFace SISA
Shards : 1
Label  : 500
🔍 Checking outputs...
✅ All outputs exist.
📈 Computing test set metrics...
Test metrics:
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
UTKFACE MULTITASK EVALUATION (TEST SET - SISA)
Container : utkface
Label     : 500
Shards    : 1
Strategy  : uniform (majority vote)
----------------------------------------------------------------------
{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
gender: acc= 88.99% | macro-prec= 89.17% | macro-recall= 88.88% | macro-f1= 88.95%
  Class metrics:
dict_keys(['Male', 'Female'])
    Male      : prec= 87.24% | recall= 92.16% | f1= 89.63% | support=2449
    Female    : prec= 91.09% | recall= 85.60% | f1= 88.26% | support=2292

{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
age   : acc

#TRƯỜNG HỢP 1 SHARD 3 SLICES

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 0

📊 DATA SUMMARY - UTKFace SISA
Shards : 1
Label  : 0
🔍 Checking outputs...
✅ All outputs exist.
📈 Computing test set metrics...
Test metrics:
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
UTKFACE MULTITASK EVALUATION (TEST SET - SISA)
Container : utkface
Label     : 0
Shards    : 1
Strategy  : uniform (majority vote)
----------------------------------------------------------------------
{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
gender: acc= 85.70% | macro-prec= 87.36% | macro-recall= 85.35% | macro-f1= 85.44%
  Class metrics:
dict_keys(['Male', 'Female'])
    Male      : prec= 80.27% | recall= 95.88% | f1= 87.38% | support=2449
    Female    : prec= 94.44% | recall= 74.83% | f1= 83.50% | support=2292

{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
age   : acc= 88

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 100

📊 DATA SUMMARY - UTKFace SISA
Shards : 1
Label  : 100
🔍 Checking outputs...
✅ All outputs exist.
📈 Computing test set metrics...
Test metrics:
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
UTKFACE MULTITASK EVALUATION (TEST SET - SISA)
Container : utkface
Label     : 100
Shards    : 1
Strategy  : uniform (majority vote)
----------------------------------------------------------------------
{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
gender: acc= 84.41% | macro-prec= 86.60% | macro-recall= 85.11% | macro-f1= 84.32%
  Class metrics:
dict_keys(['Male', 'Female'])
    Male      : prec= 97.07% | recall= 72.67% | f1= 83.12% | support=2503
    Female    : prec= 76.14% | recall= 97.54% | f1= 85.52% | support=2238

{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
age   : acc

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 1 500

📊 DATA SUMMARY - UTKFace SISA
Shards : 1
Label  : 500
🔍 Checking outputs...
✅ All outputs exist.
📈 Computing test set metrics...
Test metrics:
✅ UTKFace HDF5 loaded
Train: 18964
Test : 4741
UTKFACE MULTITASK EVALUATION (TEST SET - SISA)
Container : utkface
Label     : 500
Shards    : 1
Strategy  : uniform (majority vote)
----------------------------------------------------------------------
{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
gender: acc= 85.40% | macro-prec= 86.03% | macro-recall= 85.02% | macro-f1= 85.20%
  Class metrics:
dict_keys(['Male', 'Female'])
    Male      : prec= 82.47% | recall= 91.89% | f1= 86.92% | support=2503
    Female    : prec= 89.60% | recall= 78.15% | f1= 83.48% | support=2238

{'gender': 2, 'age': 3, 'race': 5}
{'gender': ['Male', 'Female'], 'age': ['Young', 'Middle', 'Old'], 'race': ['White', 'Black', 'Asian', 'Indian', 'Others']}
age   : acc

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving utkface_val_ovr.h5 to utkface_val_ovr.h5
Saving utkface_train_ovr.h5 to utkface_train_ovr.h5
Saving utkface_test_ovr.h5 to utkface_test_ovr.h5
Saving datasetfile_ovr to datasetfile_ovr
Saving null.time to null.time
Saving splitfile.npy to splitfile.npy
Saving requestfile_0.npy to requestfile_0.npy
Saving ovr_slices.npz to ovr_slices.npz
Saving ovr_meta.json to ovr_meta.json
User uploaded file "utkface_val_ovr.h5" with length 26449046 bytes
User uploaded file "utkface_train_ovr.h5" with length 211091735 bytes
User uploaded file "utkface_test_ovr.h5" with length 26380291 bytes
User uploaded file "datasetfile_ovr" with length 215 bytes
User uploaded file "null.time" with length 2 bytes
User uploaded file "splitfile.npy" with length 567090 bytes
User uploaded file "requestfile_0.npy" with length 277 bytes
User uploaded file "ovr_slices.npz" with length 570782 bytes
User uploaded file "ovr_meta.json" with length 479 bytes


In [ ]:
import os

# Define the target directory for UTKFace dataset files
utkface_data_dir = 'datasets/UTKFace'

# Move the other four files to the utkface_data_dir
!mv utkface_test_ovr.h5 {utkface_data_dir}/utkface_test_ovr.h5
!mv utkface_train_ovr.h5 {utkface_data_dir}/utkface_train_ovr.h5
!mv utkface_val_ovr.h5 {utkface_data_dir}/utkface_val_ovr.h5
!mv datasetfile_ovr {utkface_data_dir}/datasetfile_ovr

print(f"Moved splitfile.npy, requestfile_0.npy, ovr_slices.npz, ovr_meta.json to {utkface_data_dir}/")

Moved splitfile.npy, requestfile_0.npy, ovr_slices.npz, ovr_meta.json to datasets/UTKFace/


In [ ]:
import os

# Define the target directory for UTKFace dataset files
utkface_data_dir = 'containers/utkface_ovr'
time_dir='containers/utkface_ovr/times'

# Move the other four files to the utkface_data_dir
!mv splitfile.npy {utkface_data_dir}/splitfile.npy
!mv requestfile_0.npy {utkface_data_dir}/requestfile_0.npy
!mv ovr_slices.npz {utkface_data_dir}/ovr_slices.npz
!mv ovr_meta.json {utkface_data_dir}/ovr_meta.json
!mv null.time {time_dir}/null.time

print(f"Moved splitfile.npy, requestfile_0.npy, ovr_slices.npz, ovr_meta.json to {utkface_data_dir}/")

Moved splitfile.npy, requestfile_0.npy, ovr_slices.npz, ovr_meta.json to containers/utkface_ovr/


In [ ]:
# Move files back to the current directory from their previous incorrect location
!mv 'machine-unlearning/null (1).time' .
!mv 'machine-unlearning/splitfile (1).npy' .
!mv 'machine-unlearning/requestfile_0 (1).npy' .
!mv 'machine-unlearning/ovr_slices (1).npz' .
!mv 'machine-unlearning/ovr_meta (1).json' .

print("Các tệp đã được di chuyển trở lại thư mục hiện tại.")
print("Bây giờ, vui lòng chạy lại ô 'bd054fca' để di chuyển chúng đến 'containers/utkface_ovr' chính xác.")

Các tệp đã được di chuyển trở lại thư mục hiện tại.
Bây giờ, vui lòng chạy lại ô 'bd054fca' để di chuyển chúng đến 'containers/utkface_ovr' chính xác.


In [ ]:
%cd machine-unlearning/

/content/machine-unlearning


Khi bạn chạy đoạn mã trên, một nút "Choose Files" (Chọn tệp) sẽ xuất hiện. Nhấp vào nút đó để duyệt và chọn các tệp từ máy tính cục bộ của bạn để tải lên.

In [ ]:
!bash example-scripts/utkface-sharding/train.sh utkface_ovr 0

TRAIN UTKFACE OVR
Container : utkface_ovr
Label     : 0
Shards    : 6-9
Epochs    : 40
Batch size: 64
Optimizer : adamw
LR        : 0.0005
Dropout   : 0.2
Loss mode : auto
Focal task: race_others,age_bin2
Scheduler : 1

--------------------------------------------------------------------
Training shard=6 label=0
--------------------------------------------------------------------
✅ Đã kết nối UTKFace OVR HDF5:
   Train: 18964 samples
   Val  : 2370 samples
   Test : 2371 samples
[Shard 6][race_black] loss=bce pos_weight=4.2343
Shard 6-race_black:   0% 0/2 [00:00<?, ?it/s]
  0% 0/26 [00:00<?, ?it/s][Shard 6][Slice 0][Epoch 1] loss=105.2233 acc=79.86% val_acc=90.34%

  4% 1/26 [00:29<12:27, 29.91s/it][Shard 6][Slice 0][Epoch 2] loss=72.9358 acc=88.37% val_acc=86.29%

  8% 2/26 [00:58<11:42, 29.26s/it][Shard 6][Slice 0][Epoch 3] loss=62.1196 acc=90.10% val_acc=93.54%

 12% 3/26 [01:27<11:04, 28.88s/it][Shard 6][Slice 0][Epoch 4] loss=54.2644 acc=91.47% val_acc=93.21%

 15% 4/26 [01:55<10:

In [ ]:
!mv containers/utkface_ovr/utkface_val_ovr.h5 datasets/UTKFace/

In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh 0 utkface datasets/UTKFace/datasetfile_ver2

⚠️  Cảnh báo: Không đọc được metadata: "Unable to synchronously open attribute (can't locate attribute: 'attribute')"
✅ Đã kết nối dataset: Train=18964, Test=4741
   📊 X_train shape: (18964, 3, 64, 64)
   📊 X_test shape: (4741, 3, 64, 64)
UTKFACE MULTITASK EVALUATION
Container: utkface
Label: 0
Dataset: datasets/UTKFace/datasetfile_ver2

Shard 0 (gender) accuracy: 90.78%
Shard 1 (age) accuracy: 71.04%
Shard 2 (race) accuracy: 72.28%

Mean multitask accuracy: 78.04%
Total training time (sum shards): 8962.68s


In [ ]:
!bash example-scripts/utkface-sharding/unlearn_shard_multitask.sh utkface forget-race-slice3 2 3


Created requestfile for multitask slice unlearning
{
  "container": "utkface",
  "label": "forget-race-slice3",
  "task": "race",
  "shard": 2,
  "slice": 3,
  "forget_count": 3209,
  "mode": "overwrite"
}
✅ Unlearn request applied: shard=2, slice=3, label=forget-race-slice3


In [ ]:
!python sisa_utkface_multitask.py --train --container utkface --dataset datasets/UTKFace/datasetfile_ver2 --shard 2 --label forget-race-slice3


⚠️  Cảnh báo: Không đọc được metadata: "Unable to synchronously open attribute (can't locate attribute: 'attribute')"
✅ Đã kết nối dataset: Train=18964, Test=4741
   📊 X_train shape: (18964, 3, 64, 64)
   📊 X_test shape: (4741, 3, 64, 64)
Shard 2-race:   0% 0/4 [00:00<?, ?it/s]
  0% 0/12 [00:00<?, ?it/s][Shard 2][Slice 3][Epoch 1] loss=205.1317 acc=77.80%

  8% 1/12 [00:47<08:42, 47.46s/it][Shard 2][Slice 3][Epoch 2] loss=149.0897 acc=79.30%

 17% 2/12 [01:35<07:59, 47.95s/it][Shard 2][Slice 3][Epoch 3] loss=137.9499 acc=79.95%

 25% 3/12 [02:22<07:06, 47.42s/it][Shard 2][Slice 3][Epoch 4] loss=135.0412 acc=80.13%

 33% 4/12 [03:09<06:17, 47.18s/it][Shard 2][Slice 3][Epoch 5] loss=130.8020 acc=80.10%

 42% 5/12 [03:57<05:32, 47.43s/it][Shard 2][Slice 3][Epoch 6] loss=123.5140 acc=81.50%

 50% 6/12 [04:44<04:43, 47.25s/it][Shard 2][Slice 3][Epoch 7] loss=122.0902 acc=81.36%

 58% 7/12 [05:30<03:54, 46.93s/it][Shard 2][Slice 3][Epoch 8] loss=120.5283 acc=81.20%

 67% 8/12 [06:16<03:06, 4

In [ ]:
!cat sisa_utkface.py

"""
SISA.py - Fixed for CelebA and UTKFace
Removed ResNet50 transforms (only for CIFAR10)
"""

import numpy as np
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import Adam, SGD
from torch.nn.functional import one_hot, softmax
from sharded import sizeOfShard, getShardHash, fetchShardBatch, fetchTestBatch
import os
from glob import glob
from time import time
import json
from tqdm import tqdm
import argparse

parser = argparse.ArgumentParser()
parser.add_argument(
    "--model", default="purchase", help="Architecture to use, default purchase"
)

parser.add_argument(
    "--train", action="store_true", help="Perform SISA training on the shard"
)
parser.add_argument("--test", action="store_true", help="Compute shard predictions")

parser.add_argument(
    "--epochs",
    default=20,
    type=int,
    help="Train for the specified number of epochs, default 20",
)
parser.add_argument(
    "--batch_size",
    default=16,
    type=int,
    help="Size of the batches, releva

In [ ]:
!cat example-scripts/utkface-sharding/train.sh

#!/bin/bash
# train.sh - Train với nhiều unlearning scenarios

set -eou pipefail
IFS=$'\n\t'

shards=$1

# ===============================
# ⚙️ Cấu hình train
# ===============================
BATCH_SIZE=64
EPOCHS=30
LEARNING_RATE=0.001
OPTIMIZER=adam
CHKPT_INTERVAL=5

# Các scenarios cần train
scenarios=0

echo "======================================================================"
echo "TRAINING UTKFACE - MULTIPLE SCENARIOS"
echo "======================================================================"
echo "Shards: ${shards}"
echo "Scenarios: ${scenarios[@]}"
echo "Epochs: ${EPOCHS}"
echo "Batch size: ${BATCH_SIZE}"
echo "======================================================================"
echo ""

LOG_FILE="containers/utkface/training.log"
echo "Training started at $(date)" > "${LOG_FILE}"

global_start=$(date +%s)

# ===============================
# 🔄 Train từng scenario
# ===============================
for label in "${scenarios[@]}"; do
    
    echo ""
    echo "===========

In [ ]:
!python sisa_utkface_multitask.py \
  --test \
  --container utkface \
  --dataset datasets/UTKFace/datasetfile_ver2 \
  --shard 2 \
  --label forget-race-slice3 \
  --output_type argmax

⚠️  Cảnh báo: Không đọc được metadata: "Unable to synchronously open attribute (can't locate attribute: 'attribute')"
✅ Đã kết nối dataset: Train=18964, Test=4741
   📊 X_train shape: (18964, 3, 64, 64)
   📊 X_test shape: (4741, 3, 64, 64)


In [ ]:
!bash example-scripts/utkface-sharding/data_ver2.sh forget-race-slice3 utkface datasets/UTKFace/datasetfile_ver2

⚠️  Cảnh báo: Không đọc được metadata: "Unable to synchronously open attribute (can't locate attribute: 'attribute')"
✅ Đã kết nối dataset: Train=18964, Test=4741
   📊 X_train shape: (18964, 3, 64, 64)
   📊 X_test shape: (4741, 3, 64, 64)
UTKFACE MULTITASK EVALUATION
Container: utkface
Label: forget-race-slice3
Dataset: datasets/UTKFace/datasetfile_ver2

Traceback (most recent call last):
  File "/content/machine-unlearning/aggregation_ver2.py", line 80, in <module>
    main()
  File "/content/machine-unlearning/aggregation_ver2.py", line 50, in main
    raise FileNotFoundError(f"Missing output: {output_path}")
FileNotFoundError: Missing output: containers/utkface/outputs/shard-0:forget-race-slice3.npy


In [ ]:
!python aggregation_ver2.py \
  --label forget-race-slice3 \
  --container utkface \
  --dataset datasets/UTKFace/datasetfile_ver2

⚠️  Cảnh báo: Không đọc được metadata: "Unable to synchronously open attribute (can't locate attribute: 'attribute')"
✅ Đã kết nối dataset: Train=18964, Test=4741
   📊 X_train shape: (18964, 3, 64, 64)
   📊 X_test shape: (4741, 3, 64, 64)
UTKFACE MULTITASK EVALUATION
Container: utkface
Label: forget-race-slice3
Dataset: datasets/UTKFace/datasetfile_ver2

Traceback (most recent call last):
  File "/content/machine-unlearning/aggregation_ver2.py", line 80, in <module>
    main()
  File "/content/machine-unlearning/aggregation_ver2.py", line 50, in main
    raise FileNotFoundError(f"Missing output: {output_path}")
FileNotFoundError: Missing output: containers/utkface/outputs/shard-0:forget-race-slice3.npy
